<a href="https://colab.research.google.com/github/MInesGomes/AI-Project2026/blob/main/InesGomes_AI_BusinessProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Example dataset structure
Your customer_support.csv might look like:

Columns:


```
issue_type (e.g., “payment”, “delivery”, “technical”)
response_time (numeric, minutes)
customer_rating (1–5)
resolved (Yes/No)
priority (optional)
resolved_quickly (target: Yes/No) → e.g., resolved within 2 hours
```



For simplicity, we’ll assume:
```
Target column: resolved_quickly (values: Yes or No).
```
Other columns are features; we’ll drop obviously irrelevant ones if needed.

In [7]:
import pandas as pd
import numpy as np

# Number of rows for the dataset
n_rows = 2100

# Generate random data
data = {
    "ticket_id": np.random.randint(10000, 99999, size=n_rows),
    "customer_name": np.random.choice(['Alice', 'Bob', 'Charlie', 'David', 'Eve'], size=n_rows),
    "comment": np.random.choice(['Good', 'Bad', 'Neutral'], size=n_rows),
    'issue_type': np.random.choice(['Technical', 'Billing', 'Shipping', 'Delivery', 'Payment', 'Damaged', 'Return'], size=n_rows),
    'response_time_min': np.random.randint(5, 301, size=n_rows), # Random minutes between 5 and 300
    'customer_rating': np.random.randint(1, 6, size=n_rows), # Random rating between 1 and 5
    'resolved': np.random.choice(['Yes', 'No'], size=n_rows),
    'priority': np.random.randint(1, 6, size=n_rows), # Random priority between 1 and 5
    'target_resolved_quickly': np.random.choice(['Yes', 'No'], size=n_rows) # Random Yes/No
}

df_generated = pd.DataFrame(data);

# Save to CSV
df_generated.to_csv('customer_support.csv', index=True)

print(f"Generated a new customer_support.csv with {n_rows} rows.")
print("First 5 rows of the generated dataset:")
display(df_generated.head())

Generated a new customer_support.csv with 2100 rows.
First 5 rows of the generated dataset:


,ticket_id,customer_name,comment,issue_type,response_time_min,customer_rating,resolved,priority,target_resolved_quickly
0,73523,Charlie,Good,Technical,73,4,Yes,1,Yes
1,48155,Eve,Bad,Delivery,177,3,Yes,4,Yes
2,16485,David,Bad,Return,171,2,Yes,5,Yes
3,93045,David,Good,Technical,163,1,Yes,4,Yes
4,71892,Alice,Bad,Billing,296,5,No,1,No


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# -----------------------------
# 1. Load and explore the data
# -----------------------------
# Make sure customer_support.csv is in the same folder as this script
df = pd.read_csv("customer_support.csv")

print("First 5 rows:")
print(df.head())
print("\nColumns:", df.columns.tolist())
print("\nBasic info:")
print(df.info())
print("\nSummary statistics:")
print(df.describe(include="all"))

# -----------------------------------
# 2. Basic cleaning / preprocessing
# -----------------------------------

# Example: remove irrelevant fields if present
irrelevant_cols = ["ticket_id", "customer_name", "comment"]  # adjust to your dataset
for col in irrelevant_cols:
    if col in df.columns:
        df = df.drop(columns=[col])

# Target variable
target_col = "resolved_quickly"  # Yes/No
if target_col not in df.columns:
    raise ValueError(f"Target column '{target_col}' not found in dataset.")

# Drop rows with missing target
df = df.dropna(subset=[target_col])

# Separate features and target
X = df.drop(columns=[target_col])
y = df[target_col]

# Encode target as binary 0/1
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)  # e.g., No -> 0, Yes -> 1

# Handle categorical features
X_processed = X.copy()
categorical_cols = X_processed.select_dtypes(include=["object", "category"]).columns
numeric_cols = X_processed.select_dtypes(include=[np.number]).columns

# Label encode categorical columns (simple baseline)
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_processed[col] = le.fit_transform(X_processed[col].astype(str))
    label_encoders[col] = le

# Optionally normalize numeric columns
scaler = StandardScaler()
X_processed[numeric_cols] = scaler.fit_transform(X_processed[numeric_cols])

print("\nProcessed feature sample:")
print(X_processed.head())

# -----------------------------------
# 3. Train-test split
# -----------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)

# -----------------------------------
# 4. Build and test models
# -----------------------------------
models = {
    "DecisionTree (J48-like)": DecisionTreeClassifier(random_state=42),
    "NaiveBayes": GaussianNB(),
    "RandomForest": RandomForestClassifier(
        n_estimators=100, random_state=42
    ),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

results = []

plt.figure(figsize=(10, 8))
for name, model in models.items():
    # Train
    model.fit(X_train, y_train)
    # Predict
    y_pred = model.predict(X_test)
    y_proba = None
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        # Some models may not have predict_proba; skip ROC in that case
        y_proba = None

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    if y_proba is not None:
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        roc_auc = np.nan

    results.append({
        "model": name,
        "accuracy": acc,
        "roc_auc": roc_auc
    })

    # ROC curve
    if y_proba is not None:
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.2f})")

# Plot ROC curves
plt.plot([0, 1], [0, 1], "k--", label="Random baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for Ticket Resolution Prediction")
plt.legend()
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=150)
plt.show()

# -----------------------------------
# 5. Compare models (accuracy & ROC)
# -----------------------------------
results_df = pd.DataFrame(results)
print("\nModel comparison:")
print(results_df)

plt.figure(figsize=(8, 5))
sns.barplot(data=results_df, x="model", y="accuracy")
plt.xticks(rotation=20, ha="right")
plt.ylim(0, 1)
plt.title("Model Accuracy Comparison")
plt.tight_layout()
plt.savefig("accuracy_comparison.png", dpi=150)
plt.show()

# -----------------------------------
# 6. Confusion matrix for best model
# -----------------------------------
# Choose best model by accuracy (or ROC AUC)
best_model_name = results_df.sort_values(by="accuracy", ascending=False).iloc[0]["model"]
best_model = models[best_model_name]
y_pred_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)
cm_df = pd.DataFrame(
    cm,
    index=[f"True_{cls}" for cls in le_target.classes_],
    columns=[f"Pred_{cls}" for cls in le_target.classes_]
)

print(f"\nBest model: {best_model_name}")
print("Confusion matrix:")
print(cm_df)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title(f"Confusion Matrix - {best_model_name}")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
